In [1]:
# !pip install torch transformers ipywidgets

## Baseline Generation (No Caching)

- Investigate what happens during generation WITHOUT caching.
- Measure how long it takes.

KEY CONCEPTS:
- Tokenization: text → integer token IDs
- Forward pass: token IDs → probability distribution over next token
- Autoregressive generation: feed output back as input, one token at a time
- What does model() actually return?
- How do we pick the next token from logits?
- Why does generation get slower as the sequence gets longer?

In [2]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import warnings
warnings.filterwarnings('ignore')

Load Model - Both Tokenizer and Model Weights

In [3]:
def load_model(m_name):
    """Load Model"""
    print(f"Loading Model {m_name}")
    tokenizer = AutoTokenizer.from_pretrained(m_name)
    model = AutoModelForCausalLM.from_pretrained(m_name)
    # Put the model in inference mode
    model.eval()
    return tokenizer, model

In [4]:
model_name = "Qwen/Qwen3-0.6B"
t, m = load_model(model_name)

Loading Model Qwen/Qwen3-0.6B


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

In [5]:
m

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [6]:
t

Qwen2Tokenizer(name_or_path='Qwen/Qwen3-0.6B', vocab_size=151643, model_max_length=131072, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151647: AddedToken("<|object_ref_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151648: AddedToken("<|box_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151649: AddedToken("<|box_end|>", rstrip=False, lstrip

In [7]:
ei = t('This is a test for input sequence, let me see what the next token will ', return_tensors='pt')

In [8]:
ei['input_ids']

tensor([[1986,  374,  264, 1273,  369, 1946, 8500,   11, 1077,  752, 1490, 1128,
          279, 1790, 3950,  686,  220]])

In [9]:
ei

{'input_ids': tensor([[1986,  374,  264, 1273,  369, 1946, 8500,   11, 1077,  752, 1490, 1128,
          279, 1790, 3950,  686,  220]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [10]:
{k: v.clone() for k, v in ei.items()}

{'input_ids': tensor([[1986,  374,  264, 1273,  369, 1946, 8500,   11, 1077,  752, 1490, 1128,
           279, 1790, 3950,  686,  220]]),
 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [11]:
logits = m(**ei).logits

In [12]:
logits

tensor([[[ 5.3750,  4.4688,  4.4062,  ...,  0.8164,  0.8164,  0.8164],
         [ 6.3125,  6.2812,  2.7969,  ..., -2.5625, -2.5625, -2.5625],
         [ 6.5625,  6.5625,  1.9219,  ..., -2.7188, -2.7188, -2.7188],
         ...,
         [ 8.5000, 10.1250,  4.9688,  ..., -2.3125, -2.3125, -2.3125],
         [ 7.3750,  7.2188,  3.1250,  ..., -2.0625, -2.0625, -2.0625],
         [ 3.5156, -4.7812, -1.7266,  ..., -1.8516, -1.8516, -1.8516]]],
       dtype=torch.bfloat16, grad_fn=<UnsafeViewBackward0>)

In [13]:
torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)

tensor([[387]])

In [14]:
t.decode(torch.argmax(logits[:, -1, :], dim=-1, keepdim=True))

[' be']

In [15]:
prompt = """
Article
What is Prompt testing?


Prompt testing is the systematic process of evaluating and validating the effectiveness of prompts used in AI interactions. This practice involves assessing how well a prompt elicits the desired response from an AI model, often through a series of controlled experiments and analyses.



Understanding Prompt testing


Prompt testing is a critical step in prompt engineering that ensures prompts are performing as intended and producing high-quality, relevant outputs from AI models. It combines elements of quality assurance, performance optimization, and user experience design tailored specifically for AI interactions.

Key aspects of Prompt testing include:

Systematic Evaluation: Methodical assessment of prompt performance against predefined criteria.
Comparison Analysis: Testing multiple prompt variations to determine the most effective.
Edge Case Identification: Exploring how prompts perform in unusual or extreme scenarios.
User Simulation: Mimicking real-world usage patterns to assess prompt effectiveness.
Iterative Refinement: Using test results to inform prompt improvements.


Methods of Prompt testing


A/B Testing: Comparing two or more prompt variations to determine which performs better.
Stress Testing: Evaluating prompts under high load or challenging conditions.
Semantic Analysis: Assessing the relevance and coherence of AI responses to prompts.
User Feedback Collection: Gathering and analyzing user responses to prompt-generated outputs.
Automated Testing: Using scripts or tools to run large-scale prompt tests efficiently.
Cross-Model Testing: Evaluating prompt performance across different AI models.
Scenario-based Testing: Creating specific use cases or scenarios to test prompt effectiveness.

Advantages of Prompt testing

Improved Reliability: Ensures prompts consistently produce expected results.
Enhanced Efficiency: Identifies the most effective prompts, saving time and resources.
Better User Satisfaction: Leads to more accurate and relevant AI responses.
Risk Mitigation: Helps prevent potential issues or biases in AI outputs.
Data-Driven Optimization: Provides concrete data for informed prompt refinement.


Challenges and Considerations

Subjectivity: Difficulty in defining objective criteria for "good" prompts in some contexts.
Resource Intensity: Comprehensive testing can be time-consuming and computationally expensive.
Model Specificity: Results may vary across different AI models or versions.
Overfitting Risk: Excessive optimization for test cases may lead to reduced general performance.
Evolving AI Capabilities: Testing strategies need to adapt as AI models improve and change.


Best Practices for Prompt testing


Clear Objectives: Define specific goals and success criteria for each prompt test.
Diverse Test Sets: Use a wide range of inputs to ensure robust prompt performance.
Controlled Environment: Maintain consistent testing conditions for accurate comparisons.
Metrics Definition: Establish clear, measurable metrics for evaluating prompt effectiveness.
Version Control: Keep track of different prompt versions and their test results.
Regular Retesting: Periodically retest prompts to ensure continued effectiveness.
User Involvement: Incorporate real user testing in addition to automated methods.
Documentation: Maintain detailed records of test procedures, results, and insights.




Question:
Summarize prompt testing.
"""

Assess Naive way of generating text

In [16]:
import torch
import time

def generate_naive_qwen(model, tokenizer, prompt: str, max_new_tokens: int = 50):

    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    model = model.to(device)
    model.eval()

    # ---- Qwen3 chat formatting ----
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    model_inputs = tokenizer([text], return_tensors="pt")
    model_inputs = {k: v.to(device) for k, v in model_inputs.items()}

    generated = {k: v.clone() for k, v in model_inputs.items()}
    times_per_step = []

    with torch.no_grad():
        for step in range(max_new_tokens):
            print(f"\nStep {step+1}/{max_new_tokens}")
            step_start = time.perf_counter()

            outputs = model(
                input_ids=generated["input_ids"],
                attention_mask=generated["attention_mask"]
            )

            next_token_logits = outputs.logits[:, -1, :]
            next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)

            # append token
            generated["input_ids"] = torch.cat(
                (generated["input_ids"], next_token), dim=1
            )

            generated["attention_mask"] = torch.cat(
                (
                    generated["attention_mask"],
                    torch.ones((1, 1), dtype=torch.long, device=device)
                ),
                dim=1
            )

            times_per_step.append(time.perf_counter() - step_start)
            print(f'Step Time {(time.perf_counter() - step_start):.2f}' )

            if next_token.item() == tokenizer.eos_token_id:
                break

    # ---- Post processing
    full_ids = generated["input_ids"][0]
    prompt_len = model_inputs["input_ids"].shape[1]
    output_ids = full_ids[prompt_len:].tolist()

    decoded = tokenizer.decode(output_ids, skip_special_tokens=True)

    # Split thinking content
    try:
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0

    thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip()
    content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip()

    return thinking_content, content, times_per_step

In [17]:
start = time.perf_counter()
think_tokens, text_naive, times_naive = generate_naive_qwen(m, t, prompt, 50)
total_naive = time.perf_counter() - start
print(f"Output: {text_naive}")
print(f"Total time: {total_naive:.3f}s")
print(f"Avg per step: {sum(times_naive)/len(times_naive)*1000:.1f}ms")
print(f"First step:   {times_naive[0]*1000:.1f}ms  (slowest — MPS/GPU warmup + full prompt)")
print(f"Last step:    {times_naive[-1]*1000:.1f}ms  (faster — GPU warmed up; sequence only grew by 50 tokens)")


Step 1/50
Step Time 0.26

Step 2/50
Step Time 0.08

Step 3/50
Step Time 0.08

Step 4/50
Step Time 0.07

Step 5/50
Step Time 0.07

Step 6/50
Step Time 0.07

Step 7/50
Step Time 0.06

Step 8/50
Step Time 0.07

Step 9/50
Step Time 0.06

Step 10/50
Step Time 0.07

Step 11/50
Step Time 0.07

Step 12/50
Step Time 0.07

Step 13/50
Step Time 0.06

Step 14/50
Step Time 0.06

Step 15/50
Step Time 0.07

Step 16/50
Step Time 0.07

Step 17/50
Step Time 0.07

Step 18/50
Step Time 0.07

Step 19/50
Step Time 0.07

Step 20/50
Step Time 0.07

Step 21/50
Step Time 0.08

Step 22/50
Step Time 0.06

Step 23/50
Step Time 0.07

Step 24/50
Step Time 0.07

Step 25/50
Step Time 0.10

Step 26/50
Step Time 0.08

Step 27/50
Step Time 0.07

Step 28/50
Step Time 0.18

Step 29/50
Step Time 0.07

Step 30/50
Step Time 0.46

Step 31/50
Step Time 0.10

Step 32/50
Step Time 0.10

Step 33/50
Step Time 0.11

Step 34/50
Step Time 0.07

Step 35/50
Step Time 0.07

Step 36/50
Step Time 0.07

Step 37/50
Step Time 0.07

Step 38/5

In [18]:
def generate_cached_qwen(model, tokenizer, prompt: str, max_new_tokens: int = 50):

    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    model = model.to(device)
    model.eval()

    # ---- Qwen3 chat formatting ----
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    model_inputs = tokenizer([text], return_tensors="pt")
    model_inputs = {k: v.to(device) for k, v in model_inputs.items()}

    generated_ids = model_inputs["input_ids"]
    attention_mask = model_inputs["attention_mask"]

    past_key_values = None
    times_per_step = []

    with torch.no_grad():
        for step in range(max_new_tokens):
            print(f"\nStep {step+1}/{max_new_tokens}")
            step_start = time.perf_counter()

            if past_key_values is None:
                outputs = model(
                    input_ids=generated_ids,
                    attention_mask=attention_mask,
                    use_cache=True
                )
            else:
                outputs = model(
                    input_ids=next_token,
                    use_cache=True,
                    past_key_values=past_key_values
                )

            past_key_values = outputs.past_key_values

            next_token_logits = outputs.logits[:, -1, :]
            next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)

            generated_ids = torch.cat((generated_ids, next_token), dim=1)

            times_per_step.append(time.perf_counter() - step_start)
            print(f'Step Time {(time.perf_counter() - step_start):.2f}' )


            if next_token.item() == tokenizer.eos_token_id:
                break

    # ---- Post processing ----
    full_ids = generated_ids[0]
    prompt_len = model_inputs["input_ids"].shape[1]
    output_ids = full_ids[prompt_len:].tolist()

    decoded = tokenizer.decode(output_ids, skip_special_tokens=True)

    try:
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0

    thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip()
    content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip()

    return thinking_content, content, past_key_values, times_per_step

In [19]:
start = time.perf_counter()
thinking_content, text_cached, past_kv, times_cached = generate_cached_qwen(m, t, prompt, 50)
total_cached = time.perf_counter() - start
print(f"Output: {text_cached}")
print(f"Total time: {total_cached:.3f}s")
print(f"Avg per step: {sum(times_cached)/len(times_cached)*1000:.1f}ms")
print(f"Last step:    {times_cached[-1]*1000:.1f}ms  (same speed — only 1 token!)")
print(f"First step:   {times_cached[0]*1000:.1f}ms  (slowest — processes full prompt)")


Step 1/50
Step Time 0.08

Step 2/50
Step Time 0.06

Step 3/50
Step Time 0.02

Step 4/50
Step Time 0.02

Step 5/50
Step Time 0.02

Step 6/50
Step Time 0.02

Step 7/50
Step Time 0.02

Step 8/50
Step Time 0.02

Step 9/50
Step Time 0.02

Step 10/50
Step Time 0.02

Step 11/50
Step Time 0.02

Step 12/50
Step Time 0.02

Step 13/50
Step Time 0.02

Step 14/50
Step Time 0.02

Step 15/50
Step Time 0.02

Step 16/50
Step Time 0.02

Step 17/50
Step Time 0.01

Step 18/50
Step Time 0.02

Step 19/50
Step Time 0.02

Step 20/50
Step Time 0.01

Step 21/50
Step Time 0.01

Step 22/50
Step Time 0.04

Step 23/50
Step Time 0.02

Step 24/50
Step Time 0.02

Step 25/50
Step Time 0.02

Step 26/50
Step Time 0.01

Step 27/50
Step Time 0.02

Step 28/50
Step Time 0.02

Step 29/50
Step Time 0.02

Step 30/50
Step Time 0.02

Step 31/50
Step Time 0.02

Step 32/50
Step Time 0.02

Step 33/50
Step Time 0.02

Step 34/50
Step Time 0.02

Step 35/50
Step Time 0.02

Step 36/50
Step Time 0.02

Step 37/50
Step Time 0.02

Step 38/5

In [20]:
past_kv.layers[0].keys.shape, past_kv.layers[0].values.shape

(torch.Size([1, 8, 644, 128]), torch.Size([1, 8, 644, 128]))

In [21]:
print(f"SPEEDUP: {total_naive / total_cached:.1f}x faster with KV cache")

SPEEDUP: 12.6x faster with KV cache


In [22]:
len(past_kv.layers)

28